In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os

In [2]:
def process_and_visualize_datasets(directory_path):
    # Get all CSV files in the directory
    files = [f for f in os.listdir(directory_path) if f.endswith('.csv')]
    
    # Create a directory to save the plots if it doesn't exist
    output_dir = os.path.join(directory_path, 'output_plots')
    os.makedirs(output_dir, exist_ok=True)
    
    # Iterate through each file
    for idx, file in enumerate(files):
        file_path = os.path.join(directory_path, file)
        df = pd.read_csv(file_path)
        
        if 'datetime' not in df.columns:
            print(f"'datetime' column missing in {file}, skipping this file.")
            continue

        # Convert the datetime column to datetime format
        df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')

        if df['datetime'].isnull().all():
            print(f"Failed to parse 'datetime' column in {file}, skipping this file.")
            continue

        # Extracting the day of the month, day of the week, and hour of the day
        df['day_of_month'] = df['datetime'].dt.day
        df['day_of_week'] = df['datetime'].dt.dayofweek
        df['hour_of_day'] = df['datetime'].dt.hour
        df['month'] = df['datetime'].dt.month_name()

        # Group by the extracted intervals and count the occurrences
        day_of_month_counts = df.groupby('day_of_month')['Event'].count().reset_index()
        day_of_week_counts = df.groupby('day_of_week')['Event'].count().reset_index()
        hour_of_day_counts = df.groupby('hour_of_day')['Event'].count().reset_index()

        # Scatter plot for day of the month
        fig1 = px.scatter(day_of_month_counts, x='day_of_month', y='Event',
                          title=f'Event Count by Day of Month - {file}', labels={'day_of_month': 'Day of Month', 'Event': 'Event Count'})
        fig1.write_image(os.path.join(output_dir, f'{file}_scatter_day_of_month.png'))

        # Scatter plot for day of the week
        fig2 = px.scatter(day_of_week_counts, x='day_of_week', y='Event',
                          title=f'Event Count by Day of Week - {file}', labels={'day_of_week': 'Day of Week', 'Event': 'Event Count'})
        fig2.write_image(os.path.join(output_dir, f'{file}_scatter_day_of_week.png'))

        # Scatter plot for hour of the day
        fig3 = px.scatter(hour_of_day_counts, x='hour_of_day', y='Event',
                          title=f'Event Count by Hour of Day - {file}', labels={'hour_of_day': 'Hour of Day', 'Event': 'Event Count'})
        fig3.write_image(os.path.join(output_dir, f'{file}_scatter_hour_of_day.png'))

        # Heatmap for day of the week and hour of the day
        heatmap_data = df.groupby(['day_of_week', 'hour_of_day']).size().unstack(fill_value=0)

        fig4 = go.Figure(data=go.Heatmap(
            z=heatmap_data.values,
            x=heatmap_data.columns,
            y=heatmap_data.index,
            colorscale='YlGnBu'
        ))

        fig4.update_layout(
            title=f'Heatmap of Event Count by Day of Week and Hour of Day - {file}',
            xaxis_title='Hour of Day',
            yaxis_title='Day of Week'
        )

        fig4.write_image(os.path.join(output_dir, f'{file}_heatmap.png'))
        
        # Scatter plot for day of the month for each month separately, combining data from all years
        months = df['month'].unique()
        for month in months:
            month_data = df[df['month'] == month]
            day_of_month_counts_by_month = month_data.groupby('day_of_month')['Event'].count().reset_index()
            
            fig_month = px.scatter(day_of_month_counts_by_month, x='day_of_month', y='Event',
                                   title=f'Event Count by Day of Month - {month} - {file}', 
                                   labels={'day_of_month': 'Day of Month', 'Event': 'Event Count'})
            
            fig_month.write_image(os.path.join(output_dir, f'{file}_scatter_day_of_month_{month}.png'))

    print(f'All plots have been saved to {output_dir}')

In [3]:
# Example usage:
directory_path = 'E:\Economic_Data\Output data'
process_and_visualize_datasets(directory_path)


All plots have been saved to E:\Economic_Data\Output data\output_plots
